# Statement (P&L) Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Render an evaluated income-statement model into a review-ready page with line-item matrices, margin trends, and variance-vs-plan context.

**Prerequisites:** `04_statement_modeling/statement_modeling.ipynb` and `04_statement_modeling/statement_analytics.ipynb`.

**What you'll learn:**

- Build the minimal statement model inputs a tear sheet needs.
- Render `reporting.statement_tearsheet` inline with a reproducible generated date.
- Add a variance section without changing the underlying model.

Build a quarterly P&L model, evaluate it, and render a `statement_tearsheet` — a
single-page income-statement summary (line-item matrix, revenue/EBITDA trend,
margins) with an optional variance-vs-plan section.


In [1]:
import datetime as dt
import json

import sys
sys.path.insert(0, "..")

from _shared import demo_pl_model
from finstack_quant import reporting
from finstack_quant.statements import Evaluator
from finstack_quant.statements_analytics import VarianceConfig, run_variance

# `demo_pl_model` is the example suite's shared quarterly P&L fixture: eight
# quarters of revenue / cogs / opex, with gross profit, EBITDA, and the margin
# lines all computed in-model. `computes=` appends further in-model formulas.
base = demo_pl_model("acme", computes={"net_income": "ebitda * 0.7"})
result = Evaluator().evaluate(base)
print("EBITDA 2025Q4:", result.get("ebitda", "2025Q4"))


EBITDA 2025Q4: 38.0


## Full tear sheet

The tear sheet renders inline in Jupyter (its `_repr_html_`). Pass a fixed
`generated` date for reproducible output.

In [2]:
reporting.statement_tearsheet(result, title="Acme Corp — P&L", generated=dt.date(2026, 6, 22))

,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4
Revenue,100.00,104.00,108.00,112.00,116.00,120.00,124.00,128.00
COGS,55.00,57.00,59.00,61.00,63.00,65.00,67.00,69.00
Gross Profit,45.00,47.00,49.00,51.00,53.00,55.00,57.00,59.00
Operating Expenses,18.00,18.00,19.00,19.00,20.00,20.00,21.00,21.00
EBITDA,27.00,29.00,30.00,32.00,33.00,35.00,36.00,38.00
Net Income,18.90,20.30,21.00,22.40,23.10,24.50,25.20,26.60
Gross Margin,45.0%,45.2%,45.4%,45.5%,45.7%,45.8%,46.0%,46.1%
EBITDA Margin,27.0%,27.9%,27.8%,28.6%,28.4%,29.2%,29.0%,29.7%


## Variance vs an alternate plan

`run_variance` compares two evaluated models; pass the parsed result to the
sheet's variance section.

In [3]:
plan = demo_pl_model(
    "acme-plan",
    revenue=[100, 104, 108, 112, 120, 126, 132, 138],
    cogs=[55, 57, 59, 61, 64, 67, 70, 73],
    opex=[18, 18, 19, 19, 21, 22, 23, 24],
    computes={"net_income": "ebitda * 0.7"},
)
plan_result = Evaluator().evaluate(plan)
variance_config = VarianceConfig(
    "base",
    "plan",
    ["revenue", "ebitda"],
    ["2025Q1", "2025Q2", "2025Q3", "2025Q4"],
)
variance_report = run_variance(result, plan_result, variance_config)
variance = json.loads(variance_report.to_json())
reporting.statement_tearsheet(
    result, variance=variance, sections=["summary", "variance"], generated=dt.date(2026, 6, 22)
)

TearSheet(theme=Theme(name='institutional', font_head='Georgia,"Times New Roman",serif', font_num='"Iowan Old Style",Georgia,serif', font_sans="-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif", ink='#10243f', muted='#7a8190', pos='#1b6b4f', neg='#9b2335', accent='#7c2230', canvas='#fbfaf6', rule='#10243f', faint='#e4e1d8', grid='#cfc8b8'), title='Financial Statements', sections=[Section(title='Income Statement', body='<table class="dd"><thead><tr><th></th><th>2024Q1</th><th>2024Q2</th><th>2024Q3</th><th>2024Q4</th><th>2025Q1</th><th>2025Q2</th><th>2025Q3</th><th>2025Q4</th></tr></thead><tbody><tr><td>Revenue</td><td>100.00</td><td>104.00</td><td>108.00</td><td>112.00</td><td>116.00</td><td>120.00</td><td>124.00</td><td>128.00</td></tr><tr><td>COGS</td><td>55.00</td><td>57.00</td><td>59.00</td><td>61.00</td><td>63.00</td><td>65.00</td><td>67.00</td><td>69.00</td></tr><tr><td>Gross Profit</td><td>45.00</td><td>47.00</td><td>49.00</td><td>51.00</td><td>53.00</td><td>55.00</td><td>57.00</td><td>59.00</td></tr><tr><td>Operating Expenses</td><td>18.00</td><td>18.00</td><td>19.00</td><td>19.00</td><td>20.00</td><td>20.00</td><td>21.00</td><td>21.00</td></tr><tr><td>EBITDA</td><td>27.00</td><td>29.00</td><td>30.00</td><td>32.00</td><td>33.00</td><td>35.00</td><td>36.00</td><td>38.00</td></tr><tr><td>Net Income</td><td>18.90</td><td>20.30</td><td>21.00</td><td>22.40</td><td>23.10</td><td>24.50</td><td>25.20</td><td>26.60</td></tr><tr><td>Gross Margin</td><td>45.0%</td><td>45.2%</td><td>45.4%</td><td>45.5%</td><td>45.7%</td><td>45.8%</td><td>46.0%</td><td>46.1%</td></tr><tr><td>EBITDA Margin</td><td>27.0%</td><td>27.9%</td><td>27.8%</td><td>28.6%</td><td>28.4%</td><td>29.2%</td><td>29.0%</td><td>29.7%</td></tr></tbody></table>', subtitle=None), Section(title='Variance vs Baseline', body='<table class="dd"><thead><tr><th>Period</th><th>Metric</th><th>Baseline</th><th>Comparison</th><th>Abs Δ</th><th>% Δ</th></tr></thead><tbody><tr><td class="">2025Q1</td><td class="">revenue</td><td class="">116.00</td><td class="">120.00</td><td class="">4.00</td><td class="">+3.4%</td></tr><tr><td class="">2025Q2</td><td class="">revenue</td><td class="">120.00</td><td class="">126.00</td><td class="">6.00</td><td class="">+5.0%</td></tr><tr><td class="">2025Q3</td><td class="">revenue</td><td class="">124.00</td><td class="">132.00</td><td class="">8.00</td><td class="">+6.5%</td></tr><tr><td class="">2025Q4</td><td class="">revenue</td><td class="">128.00</td><td class="">138.00</td><td class="">10.00</td><td class="">+7.8%</td></tr><tr><td class="">2025Q1</td><td class="">ebitda</td><td class="">33.00</td><td class="">35.00</td><td class="">2.00</td><td class="">+6.1%</td></tr><tr><td class="">2025Q2</td><td class="">ebitda</td><td class="">35.00</td><td class="">37.00</td><td class="">2.00</td><td class="">+5.7%</td></tr><tr><td class="">2025Q3</td><td class="">ebitda</td><td class="">36.00</td><td class="">39.00</td><td class="">3.00</td><td class="">+8.3%</td></tr><tr><td class="">2025Q4</td><td class="">ebitda</td><td class="">38.00</td><td class="">41.00</td><td class="">3.00</td><td class="">+7.9%</td></tr></tbody></table>', subtitle=None)], eyebrow='Statement Review', subtitle='2024Q1 - 2025Q4 (8 periods)', meta_lines=['Decimal mode'], kpis=[KPI(label='Revenue', value='128.00', cls=''), KPI(label='EBITDA', value='38.00', cls=''), KPI(label='EBITDA Margin', value='29.7%', cls=''), KPI(label='Net Income', value='26.60', cls='pos')], generated=datetime.date(2026, 6, 22), footer_left='Financial Statements', footer_right='finstack-quant')

## Saving a standalone HTML file

```python
ts = reporting.statement_tearsheet(result, generated=dt.date(2026, 6, 22))
ts.save("statement_tearsheet.html")
```

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
